<a href="https://colab.research.google.com/github/jolineuichanco/DataAnalytics/blob/main/demos/Week_13_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 13 demo: Constrained Optimization for Upgrade Offers

**Context**: You work for a telecom company with 10,000 customers on a Basic plan. You want to encourage upgrades to Premium.

**Economics**:
- Premium plan: +\$40/month = **\$480/year** additional revenue
- Marketing offer: **\$50 discount** to encourage upgrade
- Budget: Only **2,500 offers** available

**The Challenge**:
- Simply picking the top 2,500 customers by value violates business constraints
- We need to maximize profit while respecting operational, strategic, and fairness constraints

**For this week**: We assume we have **perfect predictions** of upgrade probabilities (p₀ and p₁). Next week, we'll learn how to estimate these using uplift modeling.


## Setup and Data Loading

In [ ]:
import pandas as pd

# Constants
REVENUE_PER_UPGRADE = 480  # $40/month × 12 months
OFFER_COST = 50  # Marketing offer cost

url = 'https://raw.githubusercontent.com/jolineuichanco/DataAnalytics/main/demos/customer_upgrade_data_with_ground_truth.csv'
df = pd.read_csv(url)

print(f"Dataset loaded: {len(df):,} customers")
print(f"Features: {len(df.columns)} columns")
df.head()

## What if we don't do anything?

In [ ]:
print(f"\nNo Intervention: Don't send any offers")

# Calculate total upgrades
exp_total_upgrades = df['p_upgrade_no_offer'].sum()
print(f"  Expected total upgrades: {exp_total_upgrades:,.2f}")

# Calculate expected revenue, total cost
exp_revenue = exp_total_upgrades * REVENUE_PER_UPGRADE
total_cost = 0
print(f"  Expected revenue: ${exp_revenue:,.2f}")
print(f"  Total cost of offers: ${total_cost:,.2f}")

# Calculate expected profit
exp_profit = exp_revenue - total_cost
print(f"  Expected profit: ${exp_profit:,.2f}")

## What if we send everyone an offer of \$50 to upgrade?

In [ ]:
print(f"\nFull intervention: Send everyone offers")

# Calculate total upgrades
exp_total_upgrades = df['p_upgrade_with_offer'].sum()
print(f"  Expected total upgrades: {exp_total_upgrades:,.2f}")

# Calculate expected revenue, total cost
exp_revenue = exp_total_upgrades * REVENUE_PER_UPGRADE
total_cost = 10000 * OFFER_COST
print(f"  Expected revenue: ${exp_revenue:,.2f}")
print(f"  Total cost of offers: ${total_cost:,.2f}")

# Calculate expected profit
exp_profit = exp_revenue - total_cost
print(f"  Expected profit: ${exp_profit:,.2f}")

## Calculate Expected Values and Incremental Value

For each customer, we calculate:
- **EV₀**: Expected value if we DON'T send offer = p₀ × \$480
- **EV₁**: Expected value if we DO send offer = p₁ × \$480 - \$50
- **Incremental value**: EV₁ - EV₀


In [ ]:
# Calculate expected values
df['ev_no_offer'] = df['p_upgrade_no_offer'] * REVENUE_PER_UPGRADE
df['ev_with_offer'] = df['p_upgrade_with_offer'] * REVENUE_PER_UPGRADE - OFFER_COST

# Calculate incremental value
df['incremental_value'] = df['ev_with_offer'] - df['ev_no_offer']

## Unconstrained Optimization

We should send an offer to all customers with positive incremental value (i.e., EV₁ > EV₀ )

In [ ]:
# Unconstrained optimal: send offer when EV₁ > EV₀
df['select_unconstrained'] = (df['incremental_value'] > 0).astype(int)

print(f"\nUnconstrained optimal: Send offers to {df['select_unconstrained'].sum():,} customers")

# Calculate total upgrades
exp_total_upgrades = (
    df['select_unconstrained'] * df['p_upgrade_with_offer'] +
    (1 - df['select_unconstrained']) * df['p_upgrade_no_offer']
).sum()
print(f"  Expected total upgrades: {exp_total_upgrades:,.2f}")

# Calculate expected revenue, total cost
exp_revenue = exp_total_upgrades * REVENUE_PER_UPGRADE
total_cost = df['select_unconstrained'].sum() * OFFER_COST
print(f"  Expected revenue: ${exp_revenue:,.2f}")
print(f"  Total cost of offers: ${total_cost:,.2f}")

# Calculate expected profit
exp_profit = exp_revenue - total_cost
print(f"  Expected profit: ${exp_profit:,.2f}")

## What if there is a budget for max 2,500 offers?

Strategy: Sort customers by treatment effect and pick top 2,500

In [ ]:
df['select_budget'] = 0
# Get the indices of the top 2,500 customers by incremental value
# We sort the original DataFrame to ensure we get the correct original indices
top_2500_indices = df.sort_values('treatment_effect', ascending=False).head(2500).index
# Assign 1 to 'select_budget' for these top customers
df.loc[top_2500_indices, 'select_budget'] = 1

print(f"\nBudget constrained optimal: Send offers to {df['select_budget'].sum():,} customers")

# Calculate total upgrades
exp_total_upgrades = (
    df['select_budget'] * df['p_upgrade_with_offer'] +
    (1 - df['select_budget']) * df['p_upgrade_no_offer']
).sum()
print(f"  Expected total upgrades: {exp_total_upgrades:,.2f}")

# Calculate expected revenue, total cost
exp_revenue = exp_total_upgrades * REVENUE_PER_UPGRADE
total_cost = df['select_budget'].sum() * OFFER_COST
print(f"  Expected revenue: ${exp_revenue:,.2f}")
print(f"  Total cost of offers: ${total_cost:,.2f}")

# Calculate expected profit
exp_profit = exp_revenue - total_cost
print(f"  Expected profit: ${exp_profit:,.2f}")

### Visualizing Customers Selected by Budget Constraint

In [ ]:
# Treatment effect distribution: Who got selected?
fig, ax = plt.subplots(figsize=(12, 6))

# Plot histograms
ax.hist(df[df['select_budget'] == 0]['treatment_effect'],
        bins=50, alpha=0.5, label='Not selected', color='lightcoral')
ax.hist(df[df['select_budget'] == 1]['treatment_effect'],
        bins=50, alpha=0.7, label='Selected', color='green')

ax.set_xlabel('Treatment Effect (p₁ - p₀)', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Customers', fontsize=12, fontweight='bold')
ax.set_title('Treatment Effect Distribution: Selected vs Not Selected', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nTreatment Effect Summary:")
print(f"  Selected customers:     {df[df['select_budget']==1]['treatment_effect'].mean():.3f} avg")
print(f"  Not selected customers: {df[df['select_budget']==0]['treatment_effect'].mean():.3f} avg")
print(f"  Overall:                {df['treatment_effect'].mean():.3f} avg")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_budget_selected = df[df['select_budget'] == 1].copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Distribution of Customers Selected by Budget Constraint', fontsize=16, fontweight='bold')

# Plot 1: Segment Distribution
sns.countplot(y='segment', data=df_budget_selected, order=df_budget_selected['segment'].value_counts().index, ax=axes[0, 0], palette='viridis', hue='segment', legend=False)
axes[0, 0].set_title('Offers by Customer Segment', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Number of Offers')
axes[0, 0].set_ylabel('Segment')

# Plot 2: Region Distribution
sns.countplot(y='region', data=df_budget_selected, order=df_budget_selected['region'].value_counts().index, ax=axes[0, 1], palette='plasma', hue='region', legend=False)
axes[0, 1].set_title('Offers by Region', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Number of Offers')
axes[0, 1].set_ylabel('Region')

# Plot 3: Tenure Tier Distribution
sns.countplot(y='tenure_tier', data=df_budget_selected, order=df_budget_selected['tenure_tier'].value_counts().index, ax=axes[1, 0], palette='magma', hue='tenure_tier', legend=False)
axes[1, 0].set_title('Offers by Tenure Tier', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Number of Offers')
axes[1, 0].set_ylabel('Tenure Tier')

# Plot 4: Age Group Distribution
sns.countplot(y='age_group', data=df_budget_selected, order=df_budget_selected['age_group'].value_counts().index, ax=axes[1, 1], palette='cividis', hue='age_group', legend=False)
axes[1, 1].set_title('Offers by Age Group', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Number of Offers')
axes[1, 1].set_ylabel('Age Group')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

---
## Constrained Optimization with Gurobi

### Optimization Model Formulation

**Decision Variables:**
- $x_i \in \{0, 1\}$ for each customer $i$
- $x_i = 1$ if we send offer to customer $i$, 0 otherwise

**Objective:**
$$\max \sum_{i=1}^{n} \left[ x_i \cdot \text{EV}_i^{\text{with}} + (1-x_i) \cdot \text{EV}_i^{\text{no}} \right]$$

**Constraints:**
1. **Budget**: $\sum_i x_i \leq 2500$
2. **Regional capacity**: $\sum_{i \in R} x_i \leq \text{capacity}_R$ for each region $R$
3. **Segment balance**: $\sum_{i \in S} x_i \geq 375$ for each segment $S$
4. **Tenure balance**: $\sum_{i \in T} x_i \geq 250$ for each tenure tier $T$
5. **Risk management**: $\sum_{i: \text{risk}_i \geq 6} x_i \leq 500$
6. **Age fairness**: $0.30 \cdot p_A \cdot 2500 \leq \sum_{i \in A} x_i \leq 0.70 \cdot p_A \cdot 2500$ for each age group $A$
7. **VIP strategic**: $\sum_{i: \text{value}_i \geq 7} x_i \geq 0.40 \cdot |\{i: \text{value}_i \geq 7\}|$

In [ ]:
!pip install gurobipy

In [ ]:
import gurobipy as gp
from gurobipy import Env, Model, GRB

env = Env(empty=True)
env.setParam("WLSACCESSID", "xxxxxx-xxxx-xxxx-xxxx-xxxxxxxxx")  # Replace with your Access ID
env.setParam("WLSSECRET", "xxxxxx-xxxx-xxxx-xxxx-xxxxxxxx")  # Replace with your Secret Key
env.setParam("LICENSEID", 1234567)  # Replace the number with your license ID
env.start()  # Start the environment

In [ ]:
# Create Gurobi model
model = gp.Model(env=env)

# Suppress Gurobi output (set to 0 for detailed output)
model.setParam('OutputFlag', 0)

print("Creating optimization model...")

In [ ]:
# Decision variables: x[i] = 1 if customer i receives offer, 0 otherwise
customers = df.index.tolist()
x = model.addVars(customers, vtype=GRB.BINARY, name="offer")

print(f"Created {len(customers):,} decision variables")

In [ ]:
# Objective function: Maximize total expected profit
model.setObjective(
    gp.quicksum(
        x[i] * df.loc[i, 'ev_with_offer'] +
        (1 - x[i]) * df.loc[i, 'ev_no_offer']
        for i in customers
    ),
    GRB.MAXIMIZE
)

print("Objective function set: Maximize expected profit")

### Adding Constraints

In [ ]:
# CONSTRAINT 1: Budget (Hard)
model.addConstr(
    gp.quicksum(x[i] for i in customers) <= 2500,
    name="budget"
)
print("Constraint 1: Budget ≤ 2,500")

In [ ]:
# CONSTRAINT 2: Regional Capacity (Hard)

# Let's first define the regional capacity numbers
regional_capacity = {
    'Northeast': 600,
    'Southeast': 550,
    'Midwest': 500,
    'Southwest': 450,
    'West': 700
}

# Let's now add the constraints to the model
for region, capacity in regional_capacity.items():
    region_customers = df[df['region'] == region].index.tolist()
    model.addConstr(
        gp.quicksum(x[i] for i in region_customers) <= capacity,
        name=f"capacity_{region}"
    )

print(f"Constraint 2: Regional capacity ({len(regional_capacity)} regions)")

In [ ]:
# CONSTRAINT 3: Segment Balance
segments = ['Consumer', 'Small Business', 'Enterprise']
min_segment_offers = 375  # 15% of 2,500

for segment in segments:
    segment_customers = df[df['segment'] == segment].index.tolist()
    model.addConstr(
        gp.quicksum(x[i] for i in segment_customers) >= min_segment_offers,
        name=f"min_segment_{segment}"
    )

print(f"Constraint 3: Segment balance (≥375 each)")

In [ ]:
# CONSTRAINT 4: Tenure Balance
tenure_tiers = df['tenure_tier'].unique()
min_tenure_offers = 250  # 10% of 2,500

for tier in tenure_tiers:
    tier_customers = df[df['tenure_tier'] == tier].index.tolist()
    model.addConstr(
        gp.quicksum(x[i] for i in tier_customers) >= min_tenure_offers,
        name=f"min_tenure_{tier}"
    )

print(f"Constraint 4: Tenure balance (≥250 each tier)")

In [ ]:
# CONSTRAINT 5: Risk Management (Hard - maximum)
high_risk_customers = df[df['risk_score'] >= 6].index.tolist()
max_high_risk_offers = 500  # 20% of 2,500

model.addConstr(
    gp.quicksum(x[i] for i in high_risk_customers) <= max_high_risk_offers,
    name="max_high_risk"
)

print(f"Constraint 5: Risk management (≤500 high-risk customers)")

In [ ]:
# CONSTRAINT 6: Age Fairness
total_offers = 2500

for age_group in df['age_group'].unique():
    age_customers = df[df['age_group'] == age_group].index.tolist()
    proportion = len(age_customers) / len(df)
    proportional_offers = proportion * total_offers

    # Between 30% and 70% of proportional share
    min_offers = int(0.30 * proportional_offers)
    max_offers = int(0.70 * proportional_offers)

    model.addConstr(
        gp.quicksum(x[i] for i in age_customers) >= min_offers,
        name=f"age_min_{age_group}"
    )
    model.addConstr(
        gp.quicksum(x[i] for i in age_customers) <= max_offers,
        name=f"age_max_{age_group}"
    )

print(f"Constraint 6: Age fairness (30-70% proportional for each age group)")

In [ ]:
# CONSTRAINT 7: VIP Strategic
vip_customers = df[df['strategic_value'] >= 7].index.tolist()
min_vip_rate = 0.40  # At least 40% of VIPs must get offers

if len(vip_customers) > 0:
    min_vip_offers = int(min_vip_rate * len(vip_customers))
    model.addConstr(
        gp.quicksum(x[i] for i in vip_customers) >= min_vip_offers,
        name="min_vip_rate"
    )
    print(f"Constraint 7: VIP strategic (≥{min_vip_offers} of {len(vip_customers)} VIP customers)")
else:
    print("Constraint 7: No VIP customers (constraint skipped)")

### Solve the Optimization Problem

In [ ]:
print("Solving optimization model...\n")
model.optimize()

if model.status == GRB.OPTIMAL:
    print("\n" + "="*70)
    print("OPTIMAL SOLUTION FOUND")
    print("="*70)
else:
    print(f"\n  Optimization status: {model.status}")
    print("No optimal solution found. Check constraints.")

### Extract Solution

In [ ]:
# Extract solution
df['select_optimized'] = [1 if x[i].X > 0.5 else 0 for i in customers]

### Analyze Solution

In [ ]:
# Calculate optimized profit
optimized_profit = (
    df['select_optimized'] * df['ev_with_offer'] +
    (1 - df['select_optimized']) * df['ev_no_offer']
).sum()

# Summary statistics
total_selected = df['select_optimized'].sum()

print(f"\nOptimized with constraints: Send offers to {df['select_optimized'].sum():,} customers")

# Calculate total upgrades
exp_total_upgrades = (
    df['select_optimized'] * df['p_upgrade_with_offer'] +
    (1 - df['select_optimized']) * df['p_upgrade_no_offer']
).sum()
print(f"  Expected total upgrades: {exp_total_upgrades:,.2f}")

# Calculate expected revenue, total cost
exp_revenue = exp_total_upgrades * REVENUE_PER_UPGRADE
total_cost = df['select_optimized'].sum() * OFFER_COST
print(f"  Expected revenue: ${exp_revenue:,.2f}")
print(f"  Total cost of offers: ${total_cost:,.2f}")

# Calculate expected profit
exp_profit = exp_revenue - total_cost
print(f"  Expected profit: ${exp_profit:,.2f}")

### Visualizing the solution

In [ ]:
# Treatment effect distribution: Who got selected?
fig, ax = plt.subplots(figsize=(12, 6))

# Plot histograms
ax.hist(df[df['select_optimized'] == 0]['treatment_effect'],
        bins=50, alpha=0.5, label='Not selected', color='lightcoral')
ax.hist(df[df['select_optimized'] == 1]['treatment_effect'],
        bins=50, alpha=0.7, label='Selected (optimized)', color='green')

ax.set_xlabel('Treatment Effect (p₁ - p₀)', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Customers', fontsize=12, fontweight='bold')
ax.set_title('Treatment Effect Distribution: Selected vs Not Selected', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nTreatment Effect Summary:")
print(f"  Selected customers:     {df[df['select_optimized']==1]['treatment_effect'].mean():.3f} avg")
print(f"  Not selected customers: {df[df['select_optimized']==0]['treatment_effect'].mean():.3f} avg")
print(f"  Overall:                {df['treatment_effect'].mean():.3f} avg")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_opt_selected = df[df['select_optimized'] == 1].copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Distribution of Customers Selected by Optimization Model', fontsize=16, fontweight='bold')

# Plot 1: Segment Distribution
sns.countplot(y='segment', data=df_opt_selected, order=df_opt_selected['segment'].value_counts().index, ax=axes[0, 0], palette='viridis', hue='segment', legend=False)
axes[0, 0].set_title('Offers by Customer Segment', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Number of Offers')
axes[0, 0].set_ylabel('Segment')

# Plot 2: Region Distribution
sns.countplot(y='region', data=df_opt_selected, order=df_opt_selected['region'].value_counts().index, ax=axes[0, 1], palette='plasma', hue='region', legend=False)
axes[0, 1].set_title('Offers by Region', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Number of Offers')
axes[0, 1].set_ylabel('Region')

# Plot 3: Tenure Tier Distribution
sns.countplot(y='tenure_tier', data=df_opt_selected, order=df_opt_selected['tenure_tier'].value_counts().index, ax=axes[1, 0], palette='magma', hue='tenure_tier', legend=False)
axes[1, 0].set_title('Offers by Tenure Tier', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Number of Offers')
axes[1, 0].set_ylabel('Tenure Tier')

# Plot 4: Age Group Distribution
sns.countplot(y='age_group', data=df_opt_selected, order=df_opt_selected['age_group'].value_counts().index, ax=axes[1, 1], palette='cividis', hue='age_group', legend=False)
axes[1, 1].set_title('Offers by Age Group', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Number of Offers')
axes[1, 1].set_ylabel('Age Group')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()